# Latent Dimension LISA Maps

This notebook creates LISA (Local Indicators of Spatial Association) maps for latent dimensions.
It's a streamlined version of cells 29-30 from notebook 6 with minimal dependencies.

In [1]:
# Imports
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import os
import pickle
import glob
import gc
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Spatial statistics
from esda.moran import Moran_Local
from sklearn.decomposition import PCA

In [2]:
# Configuration
target_dim = 100  # Dimension to analyze
n_dims_to_map = 100  # Top N most spatial dimensions to map

# Paths
cache_dir = '../data/cache/'
working_dir = "../AE_outputs/engcensus_all/250epoch_scan_lin"
output_dir = "./plots/spatial_autocorr/"
latent_lisa_dir = f"{output_dir}/latent_lisa/"

# Create output directories
os.makedirs(latent_lisa_dir, exist_ok=True)

print(f"Target dimension: {target_dim}")
print(f"Output directory: {latent_lisa_dir}")

Target dimension: 100
Output directory: ./plots/spatial_autocorr//latent_lisa/


In [3]:
# Load cached data
print("Loading cached data...")

# Load GeoDataFrame
gdf = gpd.read_parquet(f'{cache_dir}/gdf_with_errors.parquet')
print(f"✓ Loaded GeoDataFrame: {gdf.shape}")

# Load spatial weights
with open(f'{cache_dir}/spatial_weights_queen.pkl', 'rb') as f:
    w = pickle.load(f)
w.transform = 'R'
print(f"✓ Loaded spatial weights: {w.n} observations, {w.mean_neighbors:.2f} avg neighbors")

# Load original data
df_original = pd.read_parquet(f'{cache_dir}/df_original_cached.parquet')
print(f"✓ Loaded original data: {df_original.shape}")

Loading cached data...
✓ Loaded GeoDataFrame: (188880, 17)
✓ Loaded spatial weights: 188880 observations, 5.98 avg neighbors
✓ Loaded original data: (188880, 408)


In [4]:
# Load latent representations
print(f"Loading {target_dim}-dimensional latent representations...\n")

# Load PCA latent space
pca = PCA(n_components=target_dim)
pca_latent = pca.fit_transform(df_original)
pca_latent_df = pd.DataFrame(pca_latent, index=df_original.index)
print(f"✓ PCA latent shape: {pca_latent_df.shape}")

# Load Autoencoder latent space
ae_latent_path = f"{working_dir}/census_geodemo__bottleneck_{target_dim}_v1__latent.csv"
if os.path.exists(ae_latent_path):
    ae_latent_df = pd.read_csv(ae_latent_path, index_col="OA")
    ae_latent_df = ae_latent_df.reindex(df_original.index)
    print(f"✓ AE latent shape: {ae_latent_df.shape}")
else:
    print(f"Warning: AE latent file not found at {ae_latent_path}")
    ae_latent_df = None

Loading 100-dimensional latent representations...

✓ PCA latent shape: (188880, 100)
✓ AE latent shape: (188880, 100)


In [5]:
# Load or compute Moran's I for latent dimensions
data_dir = f"{output_dir}/data/"
latent_morans_path = f"{data_dir}/latent_morans.csv"

if os.path.exists(latent_morans_path):
    latent_morans_df = pd.read_csv(latent_morans_path)
    print(f"✓ Loaded latent Moran's I results from {latent_morans_path}")
else:
    print("Computing Moran's I for each latent dimension...")
    print("This may take a while...\n")
    
    from esda.moran import Moran
    
    latent_morans = []
    
    # PCA
    for i in tqdm(range(target_dim), desc="PCA dimensions"):
        moran = Moran(pca_latent_df.iloc[:, i].values, w)
        latent_morans.append({
            'dimension_idx': i,
            'method': 'PCA',
            'morans_i': moran.I,
            'p_value': moran.p_sim
        })
    
    # Autoencoder
    if ae_latent_df is not None:
        for i in tqdm(range(target_dim), desc="AE dimensions"):
            moran = Moran(ae_latent_df.iloc[:, i].values, w)
            latent_morans.append({
                'dimension_idx': i,
                'method': 'Autoencoder',
                'morans_i': moran.I,
                'p_value': moran.p_sim
            })
    
    latent_morans_df = pd.DataFrame(latent_morans)
    
    # Save results
    os.makedirs(data_dir, exist_ok=True)
    latent_morans_df.to_csv(latent_morans_path, index=False)
    print(f"\n✓ Saved latent Moran's I results to {latent_morans_path}")

✓ Loaded latent Moran's I results from ./plots/spatial_autocorr//data//latent_morans.csv


In [6]:
# Identify top most spatial dimensions
print("="*80)
print(f"CREATING LISA MAPS FOR LATENT DIMENSIONS ({target_dim}D)")
print("="*80)

# Get top N most spatial dimensions
pca_dims = latent_morans_df[latent_morans_df['method'] == 'PCA'].sort_values('morans_i', ascending=False)
top_pca_dims = pca_dims.head(n_dims_to_map)['dimension_idx'].values

if ae_latent_df is not None:
    ae_dims = latent_morans_df[latent_morans_df['method'] == 'Autoencoder'].sort_values('morans_i', ascending=False)
    top_ae_dims = ae_dims.head(n_dims_to_map)['dimension_idx'].values
else:
    top_ae_dims = []

print(f"\nMost spatial PCA dimensions (top 10):")
for i, idx in enumerate(top_pca_dims[:10], 1):
    mi = pca_dims[pca_dims['dimension_idx'] == idx]['morans_i'].values[0]
    print(f"  #{i}: Dim {idx} (Moran's I = {mi:.4f})")

if len(top_ae_dims) > 0:
    print(f"\nMost spatial AE dimensions (top 10):")
    for i, idx in enumerate(top_ae_dims[:10], 1):
        mi = ae_dims[ae_dims['dimension_idx'] == idx]['morans_i'].values[0]
        print(f"  #{i}: Dim {idx} (Moran's I = {mi:.4f})")

# Save rankings to CSV
pca_dims.to_csv(f"{latent_lisa_dir}/pca_ranking_{target_dim}d.csv", index=False)
if len(top_ae_dims) > 0:
    ae_dims.to_csv(f"{latent_lisa_dir}/ae_ranking_{target_dim}d.csv", index=False)
print(f"\n✓ Saved rankings to {latent_lisa_dir}")

CREATING LISA MAPS FOR LATENT DIMENSIONS (100D)

Most spatial PCA dimensions (top 10):
  #1: Dim 0 (Moran's I = 0.7507)
  #2: Dim 1 (Moran's I = 0.6583)
  #3: Dim 12 (Moran's I = 0.6332)
  #4: Dim 2 (Moran's I = 0.6268)
  #5: Dim 7 (Moran's I = 0.5465)
  #6: Dim 9 (Moran's I = 0.5385)
  #7: Dim 10 (Moran's I = 0.5375)
  #8: Dim 17 (Moran's I = 0.5279)
  #9: Dim 11 (Moran's I = 0.5209)
  #10: Dim 5 (Moran's I = 0.5204)

Most spatial AE dimensions (top 10):
  #1: Dim 53 (Moran's I = 0.8410)
  #2: Dim 16 (Moran's I = 0.8052)
  #3: Dim 26 (Moran's I = 0.7941)
  #4: Dim 69 (Moran's I = 0.7678)
  #5: Dim 47 (Moran's I = 0.7619)
  #6: Dim 36 (Moran's I = 0.7556)
  #7: Dim 31 (Moran's I = 0.7440)
  #8: Dim 29 (Moran's I = 0.7306)
  #9: Dim 62 (Moran's I = 0.7278)
  #10: Dim 70 (Moran's I = 0.7229)

✓ Saved rankings to ./plots/spatial_autocorr//latent_lisa/


## Cell 29: Compute LISA for Latent Dimensions

In [ ]:
def compute_lisa(dim_idx, latent_df, dims_df, method, rank, output_dir, oa_codes, w):
    """Compute LISA and save to CSV only"""
    mi = dims_df[dims_df['dimension_idx'] == dim_idx]['morans_i'].values[0]
    latent_values = latent_df.iloc[:, dim_idx].values
    
    lisa = Moran_Local(latent_values, w, permutations=99)
    
    significant = lisa.p_sim < 0.05
    clusters = lisa.q.copy()
    clusters[~significant] = 0
    
    cluster_labels = {0: 'Not sig', 1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}
    
    df = pd.DataFrame({
        'OA': oa_codes,
        'value': latent_values,
        'local_i': lisa.Is,
        'p_value': lisa.p_sim,
        'cluster': clusters,
        'cluster_label': [cluster_labels[c] for c in clusters],
        'morans_i': mi
    })
    
    df.to_csv(f"{output_dir}/{method.lower()}_dim{dim_idx:02d}_rank{rank:02d}.csv", index=False)
    
    n_sig = significant.sum()
    
    del lisa, df, latent_values, clusters, significant
    gc.collect()
    
    return n_sig

# Setup
oa_codes = gdf.reset_index()['OA'].values

if ae_latent_df is not None and len(top_ae_dims) > 0:
    print("\nProcessing AE dimensions...")
    for rank, dim_idx in enumerate(tqdm(top_ae_dims), 1):
        n_sig = compute_lisa(dim_idx, ae_latent_df, ae_dims, 'AE', rank, latent_lisa_dir, oa_codes, w)

print("\nProcessing PCA dimensions...")
for rank, dim_idx in enumerate(tqdm(top_pca_dims), 1):
    n_sig = compute_lisa(dim_idx, pca_latent_df, pca_dims, 'PCA', rank, latent_lisa_dir, oa_codes, w)

print(f"\n✓ All CSVs saved to {latent_lisa_dir}")


Processing AE dimensions...


100%|██████████| 100/100 [48:43<00:00, 29.23s/it]



Processing PCA dimensions...


100%|██████████| 100/100 [47:51<00:00, 28.72s/it]


✓ All CSVs saved to ./plots/spatial_autocorr//latent_lisa/


: 

## Cell 30: Plot LISA Maps from CSVs

In [4]:
def plot_lisa(csv_path, base_gdf, output_dir):
    """Load CSV and plot LISA map"""
    df = pd.read_csv(csv_path)
    
    # Create colour array aligned to base_gdf order
    cluster_map = dict(zip(df['OA'], df['cluster']))
    clusters = base_gdf['OA'].map(cluster_map)
    
    cluster_colors = {0: 'lightgray', 1: '#d7191c', 2: '#fdae61', 3: '#2c7bb6', 4: '#abd9e9'}
    colors = clusters.map(cluster_colors).values
    
    # Extract info for title
    filename = os.path.basename(csv_path).replace('.csv', '')
    mi = df['morans_i'].iloc[0]
    n_sig = (df['cluster'] != 0).sum()
    
    fig, ax = plt.subplots(figsize=(10, 10))
    base_gdf.plot(color=colors, ax=ax, linewidth=0, edgecolor='none')
    
    ax.set_title(f"{filename}\nMoran's I = {mi:.4f} | {n_sig:,} significant", fontsize=12)
    ax.axis('off')
    
    legend = [
        Patch(facecolor='#d7191c', label='HH'), 
        Patch(facecolor='#2c7bb6', label='LL'),
        Patch(facecolor='#fdae61', label='LH'), 
        Patch(facecolor='#abd9e9', label='HL'),
        Patch(facecolor='lightgray', label='Not sig')
    ]
    ax.legend(handles=legend, loc='lower left', fontsize=9)
    
    plt.savefig(f"{output_dir}/{filename}.png", dpi=300, bbox_inches='tight')
    plt.close('all')
    
    del df, clusters, colors, fig, ax
    gc.collect()

In [ ]:
csv_files = sorted(glob.glob(f"{latent_lisa_dir}/*.csv"))
# Filter out ranking files
csv_files = [f for f in csv_files if 'ranking' not in f]
print(f"Plotting {len(csv_files)} maps...")
base_gdf = gdf.reset_index()[['OA', 'geometry']].copy()
for csv_path in tqdm(csv_files):
    plot_lisa(csv_path, base_gdf, latent_lisa_dir)

Plotting 200 maps...


  0%|          | 0/200 [00:00<?, ?it/s]


NameError: name 'base_gdf' is not defined